# Undecidable Problems — The Tetris Proof

Some problems have no algorithm that can always solve them. Not because we haven't found the right code yet — but because it is **mathematically impossible** for any such algorithm to exist.

This lesson covers:
- What a **decision problem** is
- What **decidability** means
- The **Halting Problem** — the most famous undecidable problem in computer science
- How **Tetris** was proven to be undecidable
- A live playable Tetris game built with the OCS Game Engine

---

## 1. Decision Problems

A **decision problem** is any question that has a yes/no answer.

Examples:
- Is 17 a prime number? → **Yes**
- Does this list contain duplicates? → **Yes or No**
- Will this program ever stop running? → **???**

The first two are easy — we can write an algorithm to answer them correctly every time for any input. They are **decidable**.

The third one turns out to be **undecidable** — no algorithm can correctly answer it for all possible programs.

---

## 2. What Does "Decidable" Mean?

A problem is **decidable** if there exists an algorithm that:
1. Always terminates (doesn't run forever)
2. Always gives the correct yes/no answer
3. Works for **every possible input** — not just easy ones

A problem is **undecidable** if no such algorithm can ever exist — even with unlimited time and memory.

This is different from a problem being **hard**. Hard problems take a long time but can still be solved. Undecidable problems **cannot be solved algorithmically at all**.

| | Solvable? | Always terminates? | Example |
|---|---|---|---|
| **Decidable** | ✅ | ✅ | Is N prime? |
| **Undecidable** | ❌ | N/A | Does program P halt? |
| **Hard but decidable** | ✅ | ✅ (slowly) | Traveling salesman |

---

## 3. The Halting Problem

Proposed by **Alan Turing in 1936**, the Halting Problem asks:

> *Given any program P and any input I, will P eventually stop — or run forever?*

Turing proved that **no algorithm can answer this for all programs**. Here's the intuition:

### Proof by Contradiction

Assume a magic function `halts(P, I)` exists that returns `True` if program P halts on input I, and `False` if it runs forever.

Now build this program:

```python
def trick(P):
    if halts(P, P):   # If P halts when given itself as input...
        loop_forever() # ...then loop forever
    else:              # If P runs forever...
        return         # ...then stop
```

Now ask: **what does `trick(trick)` do?**

- If `trick(trick)` halts → `halts(trick, trick)` returns `True` → `trick` loops forever. **Contradiction.**
- If `trick(trick)` loops → `halts(trick, trick)` returns `False` → `trick` stops. **Contradiction.**

Either way we get a contradiction. Therefore `halts` **cannot exist**.

The Halting Problem is **undecidable**. ✅

---

In [ ]:
# Python demo — programs that halt vs. loop forever
# We can detect simple cases, but NOT the general case.

def might_halt_example_1():
    """This clearly halts."""
    return 42

def might_halt_example_2(n):
    """This halts only if n > 0."""
    while n > 0:
        n -= 1
    return n

def might_halt_example_3():
    """This NEVER halts."""
    x = 0
    while True:
        x += 1  # runs forever

def might_halt_collatz(n):
    """
    The Collatz conjecture: start with any n.
    If even: divide by 2. If odd: multiply by 3, add 1.
    Does this always reach 1? Nobody knows for certain.
    This is an open problem in mathematics.
    """
    steps = 0
    while n != 1:
        if n % 2 == 0:
            n //= 2
        else:
            n = 3 * n + 1
        steps += 1
        if steps > 10_000_000:
            return f"Gave up after {steps} steps — still hasn't reached 1"
    return f"Reached 1 in {steps} steps"

print("Example 1 halts:", might_halt_example_1())
print("Example 2 (n=5) halts:", might_halt_example_2(5))
print("Collatz(27):", might_halt_collatz(27))
print("Collatz(871):", might_halt_collatz(871))
print()
print("We can check specific cases — but no algorithm can check ALL programs.")

## 4. So What Does This Have to Do With Tetris?

In **2002**, Erik Demaine, Susan Hohenberger, and David Liben-Nowell published a paper:

> *"Tetris is Hard, Even to Approximate"*

They proved that **generalized Tetris** (with an infinite board and an infinite sequence of pieces) is **undecidable** in the following sense:

### The Survival Question

> *Given a starting board configuration and an infinite sequence of incoming pieces, is there any strategy that keeps the player alive forever?*

**This question is undecidable.** No algorithm can answer it correctly for all possible board + piece combinations.

### Why?

They showed Tetris can **simulate arbitrary computation** — you can encode any computation into a Tetris board. Since the Halting Problem is undecidable, and Tetris can simulate arbitrary programs, the survival question in Tetris reduces to the Halting Problem.

### Real-World Implications

Even the **finite** version has hard problems:
- Maximizing rows cleared: **NP-complete**
- Surviving a given piece sequence: **NP-complete**
- Placing pieces with minimum holes: **NP-complete**

This means no perfect Tetris AI can exist that's guaranteed to solve every sequence optimally.

---

## 5. Play It — Experience the Undecidability

The game below is built with the **OCS Game Engine v1.1**. Each piece that falls represents an "input" to the system. No strategy guarantees survival forever — that's not a flaw in the game, it's a theorem.

**Controls:**
- `← →` — Move left/right
- `↑` — Rotate
- `↓` — Soft drop
- `Space` — Hard drop
- `R` — Restart

As you play, think about: *Can you guarantee you'll never lose? Why or why not?*

In [ ]:
%%js
// GAME_RUNNER: Tetris — Undecidability in Action | panel: tetris_panel, slot: right, layout: row, ratio: 35-65, height: 560px, width: 100%

import GameControl from '/assets/js/GameEnginev1.1/essentials/GameControl.js';

// ── Tetromino definitions ────────────────────────────────────────────────────
const PIECES = [
  { shape: [[1,1,1,1]],                color: '#00f0f0', name: 'I' },
  { shape: [[1,1],[1,1]],              color: '#f0f000', name: 'O' },
  { shape: [[0,1,0],[1,1,1]],          color: '#a000f0', name: 'T' },
  { shape: [[0,1,1],[1,1,0]],          color: '#00f000', name: 'S' },
  { shape: [[1,1,0],[0,1,1]],          color: '#f00000', name: 'Z' },
  { shape: [[1,0,0],[1,1,1]],          color: '#0000f0', name: 'J' },
  { shape: [[0,0,1],[1,1,1]],          color: '#f0a000', name: 'L' },
];

const COLS = 10, ROWS = 20;

// ── TetrisGame GameObject ────────────────────────────────────────────────────
class TetrisGame {
  constructor(data, gameEnv) {
    this.gameEnv = gameEnv;
    this.canvas  = document.createElement('canvas');
    this.canvas.id = 'tetris-main';
    this.ctx = this.canvas.getContext('2d');
    this.canvas.style.cssText = 'position:absolute;top:0;left:0;z-index:2;';
    gameEnv.container.appendChild(this.canvas);
    gameEnv.tetrisGame = this;
    this._init();
    this.resize();
  }

  _init() {
    this.board       = Array.from({length: ROWS}, () => Array(COLS).fill(null));
    this.score       = 0;
    this.lines       = 0;
    this.level       = 1;
    this.gameOver    = false;
    this.paused      = false;
    this.dropMs      = 800;
    this.lastDrop    = performance.now();
    this.flashFrames = 0;
    this.nextPiece   = this._makePiece();
    this._spawn();
  }

  _makePiece() {
    const t = PIECES[Math.floor(Math.random() * PIECES.length)];
    return { shape: t.shape.map(r => [...r]), color: t.color, name: t.name };
  }

  _spawn() {
    const t = this.nextPiece;
    this.piece = {
      shape: t.shape.map(r => [...r]),
      color: t.color,
      name:  t.name,
      x: Math.floor(COLS / 2) - Math.floor(t.shape[0].length / 2),
      y: 0,
    };
    this.nextPiece = this._makePiece();
    if (this._hits(this.piece)) this.gameOver = true;
  }

  _hits(p) {
    for (let r = 0; r < p.shape.length; r++)
      for (let c = 0; c < p.shape[r].length; c++) {
        if (!p.shape[r][c]) continue;
        const nx = p.x + c, ny = p.y + r;
        if (nx < 0 || nx >= COLS || ny >= ROWS) return true;
        if (ny >= 0 && this.board[ny][nx]) return true;
      }
    return false;
  }

  _lock() {
    for (let r = 0; r < this.piece.shape.length; r++)
      for (let c = 0; c < this.piece.shape[r].length; c++) {
        if (!this.piece.shape[r][c]) continue;
        const ny = this.piece.y + r;
        if (ny < 0) { this.gameOver = true; return; }
        this.board[ny][this.piece.x + c] = this.piece.color;
      }
    this._clearLines();
    this._spawn();
  }

  _clearLines() {
    const pts = [0, 100, 300, 500, 800];
    let cleared = 0;
    for (let r = ROWS - 1; r >= 0; r--) {
      if (this.board[r].every(c => c !== null)) {
        this.board.splice(r, 1);
        this.board.unshift(Array(COLS).fill(null));
        cleared++; r++;
      }
    }
    if (cleared) {
      this.score += (pts[Math.min(cleared, 4)] || 800) * this.level;
      this.lines += cleared;
      this.level  = Math.floor(this.lines / 10) + 1;
      this.dropMs = Math.max(80, 800 - (this.level - 1) * 72);
      this.flashFrames = 8;
    }
  }

  _rotate(shape) { return shape[0].map((_, i) => shape.map(row => row[i]).reverse()); }

  _ghostY() {
    let gy = this.piece.y;
    while (!this._hits({...this.piece, y: gy + 1})) gy++;
    return gy;
  }

  moveLeft()  { const p = {...this.piece, x: this.piece.x - 1}; if (!this._hits(p)) this.piece.x--; }
  moveRight() { const p = {...this.piece, x: this.piece.x + 1}; if (!this._hits(p)) this.piece.x++; }
  softDrop()  { const p = {...this.piece, y: this.piece.y + 1}; if (!this._hits(p)) { this.piece.y++; this.lastDrop = performance.now(); } else this._lock(); }
  hardDrop()  { this.piece.y = this._ghostY(); this._lock(); }
  rotate()    { const s = this._rotate(this.piece.shape); if (!this._hits({...this.piece, shape: s})) this.piece.shape = s; }

  update() {
    if (this.gameOver || this.paused) { this.draw(); return; }
    const now = performance.now();
    if (now - this.lastDrop >= this.dropMs) {
      this.softDrop();
      this.lastDrop = now;
    }
    if (this.flashFrames > 0) this.flashFrames--;
    this.draw();
  }

  draw() {
    const C = this.CELL;
    const BW = COLS * C;
    const BH = ROWS * C;
    const HW = 110;
    const W  = BW + HW;
    const ctx = this.ctx;

    // Background
    ctx.fillStyle = '#0a0a18';
    ctx.fillRect(0, 0, W, BH);

    // Flash on line clear
    if (this.flashFrames > 0) {
      ctx.fillStyle = 'rgba(255,255,255,' + (this.flashFrames / 8 * 0.15) + ')';
      ctx.fillRect(0, 0, BW, BH);
    }

    // Grid lines
    ctx.strokeStyle = 'rgba(255,255,255,0.04)';
    ctx.lineWidth = 1;
    for (let r = 0; r <= ROWS; r++) { ctx.beginPath(); ctx.moveTo(0, r*C); ctx.lineTo(BW, r*C); ctx.stroke(); }
    for (let c = 0; c <= COLS; c++) { ctx.beginPath(); ctx.moveTo(c*C, 0); ctx.lineTo(c*C, BH); ctx.stroke(); }

    // Board border
    ctx.strokeStyle = 'rgba(255,51,51,0.4)';
    ctx.lineWidth = 2;
    ctx.strokeRect(0, 0, BW, BH);

    // Locked cells
    for (let r = 0; r < ROWS; r++)
      for (let c = 0; c < COLS; c++)
        if (this.board[r][c]) this._cell(ctx, c, r, this.board[r][c], C);

    if (!this.gameOver && this.piece) {
      // Ghost
      const gy = this._ghostY();
      ctx.fillStyle = 'rgba(255,255,255,0.08)';
      this.piece.shape.forEach((row, r) =>
        row.forEach((v, c) => {
          if (v && gy + r >= 0)
            ctx.fillRect((this.piece.x+c)*C+1, (gy+r)*C+1, C-2, C-2);
        })
      );

      // Active piece
      this.piece.shape.forEach((row, r) =>
        row.forEach((v, c) => { if (v && this.piece.y+r >= 0) this._cell(ctx, this.piece.x+c, this.piece.y+r, this.piece.color, C); })
      );
    }

    // HUD panel
    const hx = BW + 8;
    ctx.fillStyle = 'rgba(255,255,255,0.04)';
    ctx.fillRect(BW, 0, HW, BH);

    const label = (txt, x, y) => { ctx.fillStyle='rgba(255,255,255,0.45)'; ctx.font='10px monospace'; ctx.fillText(txt, x, y); };
    const value = (txt, x, y) => { ctx.fillStyle='#fff'; ctx.font='bold 15px monospace'; ctx.fillText(txt, x, y); };

    label('SCORE', hx, 24);  value(this.score, hx, 42);
    label('LINES', hx, 68);  value(this.lines, hx, 86);
    label('LEVEL', hx, 112); value(this.level, hx, 130);

    // Next piece preview
    label('NEXT', hx, 158);
    if (this.nextPiece) {
      const pc = 12;
      this.nextPiece.shape.forEach((row, r) =>
        row.forEach((v, c) => {
          if (!v) return;
          ctx.fillStyle = this.nextPiece.color;
          ctx.fillRect(hx + c*pc, 165 + r*pc, pc-1, pc-1);
        })
      );
    }

    // Controls
    label('CONTROLS', hx, 230);
    ctx.fillStyle='rgba(255,255,255,0.35)'; ctx.font='9px monospace';
    ['← → move','↑ rotate','↓ soft','SPC hard','R restart'].forEach((t,i) => ctx.fillText(t, hx, 244+i*13));

    // Game over overlay
    if (this.gameOver) {
      ctx.fillStyle = 'rgba(0,0,0,0.75)';
      ctx.fillRect(0, 0, BW, BH);
      ctx.fillStyle = '#ff3333';
      ctx.font = 'bold 18px monospace';
      ctx.textAlign = 'center';
      ctx.fillText('GAME OVER', BW/2, BH/2 - 22);
      ctx.fillStyle = '#fff';
      ctx.font = '13px monospace';
      ctx.fillText('Score: ' + this.score, BW/2, BH/2 + 2);
      ctx.fillText('Press R to restart', BW/2, BH/2 + 22);
      ctx.textAlign = 'left';
    }
  }

  _cell(ctx, c, r, color, C) {
    ctx.fillStyle = color;
    ctx.fillRect(c*C+1, r*C+1, C-2, C-2);
    ctx.fillStyle = 'rgba(255,255,255,0.28)';
    ctx.fillRect(c*C+1, r*C+1, C-2, 3);
    ctx.fillRect(c*C+1, r*C+1, 3, C-2);
    ctx.fillStyle = 'rgba(0,0,0,0.2)';
    ctx.fillRect(c*C+1, r*C + C-3, C-2, 2);
  }

  resize() {
    const HUD_W = 110;
    const H = (this.gameEnv.innerHeight || 520) - 50;
    const containerW = this.gameEnv.container?.clientWidth
                    || this.gameEnv.container?.parentElement?.clientWidth
                    || this.gameEnv.innerWidth
                    || 600;
    const W = containerW - HUD_W;
    const cellByH = Math.floor(H / ROWS);
    const cellByW = Math.floor(W / COLS);
    this.CELL = Math.max(16, Math.min(cellByH, cellByW));
    const BW  = COLS * this.CELL;
    const BH  = ROWS * this.CELL;
    this.canvas.width  = BW + HUD_W;
    this.canvas.height = BH;
    this.canvas.style.cssText = 'position:absolute;top:6px;left:50%;transform:translateX(-50%);z-index:2;';
  }

  destroy() { if (this.canvas.parentNode) this.canvas.parentNode.removeChild(this.canvas); }
  collisionChecks() {}
  handleCollisionState() {}
}

// ── TetrisLevel ───────────────────────────────────────────────────────────────
class TetrisLevel {
  constructor(gameEnv) {
    this.gameEnv = gameEnv;
    this.classes = [{ class: TetrisGame, data: {} }];
  }

  initialize() {
    this._key = (e) => {
      const g = this.gameEnv.tetrisGame;
      if (!g) return;
      if (e.key === 'r' || e.key === 'R') { g._init(); return; }
      if (g.gameOver || g.paused) return;
      if (['ArrowLeft','ArrowRight','ArrowUp','ArrowDown',' '].includes(e.key)) e.preventDefault();
      if (e.key === 'ArrowLeft')  g.moveLeft();
      if (e.key === 'ArrowRight') g.moveRight();
      if (e.key === 'ArrowUp')    g.rotate();
      if (e.key === 'ArrowDown')  g.softDrop();
      if (e.key === ' ')          g.hardDrop();
    };
    document.addEventListener('keydown', this._key);
  }

  destroy() { document.removeEventListener('keydown', this._key); }
}

export const gameLevelClasses = [TetrisLevel];
export { GameControl };

In [ ]:
%%html
<!-- UI_RUNNER: Undecidability info panel | panel: tetris_panel, slot: left, layout: row, ratio: 35-65, gap: 1rem -->
<div style="padding:1rem;font-family:sans-serif;">
  <h3 style="margin:0 0 0.75rem;color:#c00;">Why This Game is Undecidable</h3>
  <p style="font-size:0.875rem;line-height:1.6;margin-bottom:0.75rem;">
    The piece sequence coming at you is essentially a stream of inputs. No algorithm can examine that stream and guarantee a winning strategy exists — the question of survival is formally equivalent to the Halting Problem.
  </p>
  <hr style="margin:0.75rem 0;opacity:0.2;">
  <p style="font-size:0.8rem;color:#555;line-height:1.6;"><strong>Demaine et al. (2002)</strong> proved:</p>
  <ul style="font-size:0.8rem;color:#444;line-height:1.8;padding-left:1.2rem;">
    <li>Maximizing rows cleared: <strong>NP-complete</strong></li>
    <li>Surviving a piece sequence: <strong>NP-complete</strong></li>
    <li>Survival on infinite board: <strong>Undecidable</strong></li>
  </ul>
  <hr style="margin:0.75rem 0;opacity:0.2;">
  <p style="font-size:0.8rem;color:#555;">The game uses the <strong>OCS Game Engine v1.1</strong>. The <code>TetrisGame</code> class extends <code>GameObject</code>; the game loop is controlled by <code>GameControl</code>.</p>
</div>

---

## 6. Other Games With Undecidable Problems

Tetris isn't alone. Several classic games have been proven to involve undecidable or NP-hard problems:

| Game | Problem | Result |
|---|---|---|
| **Tetris** | Survive infinite piece sequence | Undecidable |
| **Minecraft** | Redstone circuit halting | Undecidable (Turing-complete) |
| **Super Mario Bros.** | Generalized level completion | PSPACE-complete |
| **Minesweeper** | Consistent board solution | NP-complete |
| **Pokemon TCG** | Optimal game state | Undecidable |
| **Magic: The Gathering** | Game resolution | Undecidable (2019) |

The pattern: games that are **expressive enough to simulate computation** inherit the undecidability of the Halting Problem.

---

## 7. Discussion Questions

Think about and answer the following:

1. **In your own words**, what is the difference between a problem being *hard* and a problem being *undecidable*?

2. The Halting Problem proof uses a program that analyzes itself. Why does self-reference create a contradiction?

3. Tetris can be used to simulate arbitrary computation. What does that tell you about how expressive the game's rules are?

4. If no perfect Tetris AI can exist, why do some AIs still play Tetris extremely well? What are they optimizing for?

5. Give one real-world example (outside of games) where the Halting Problem has practical implications for software developers.

---

In [ ]:
# Reflection: write your answers here as Python comments or strings

q1 = """
Your answer here...
"""

q2 = """
Your answer here...
"""

q3 = """
Your answer here...
"""

q4 = """
Your answer here...
"""

q5 = """
Your answer here...
"""

print("Answers saved.")

## Summary

| Concept | Definition |
|---|---|
| **Decision Problem** | A yes/no question that an algorithm must answer |
| **Decidable** | A problem solvable by an algorithm that always terminates and is always correct |
| **Undecidable** | A problem for which no such algorithm can ever exist |
| **Halting Problem** | Given program P and input I, does P halt? — proven undecidable by Turing (1936) |
| **Tetris Undecidability** | Survival on an infinite board is undecidable — proven by Demaine et al. (2002) |

Undecidability is not a temporary limitation — it is a fundamental boundary of what computation can do. Some questions, no matter how powerful the computer, simply cannot always be answered correctly by any algorithm.

---
*Lesson by Nicolas Diaz · AP CSP · Del Norte High School · Spring 2026*